# Zodiac Crib Analysis

## Hypothesis
The zodiac pages (f70v2–f73v) contain illustrations of zodiac signs with labels next to
each nymph figure (~30 per sign = degrees). If any word on these pages encodes the name
of the zodiac sign, it could serve as a **crib** (known plaintext–ciphertext pair) to
constrain or validate decipherment mappings.

## Approach
1. Parse the corpus preserving folio and line-type metadata
2. Separate labels (`@Lz`/`&Lz`) from circular text (`@Cc`)
3. Identify words unique to each zodiac folio
4. Test crib compatibility with Hebrew zodiac names under glyph→char mappings
5. Check cross-sign consistency of any viable crib assignments

In [1]:
# ── Cell 1: Imports + Corpus Parser ────────────────────────────────────
import re, json, math
from collections import Counter, defaultdict
from itertools import combinations, product
import numpy as np

# ── Parser that preserves folio + line type ──
def parse_sta_with_folios(filepath):
    """Parse STA1 corpus returning list of (folio, line_id, line_type, [words]).
    Each word is a list of glyph bigrams (e.g. ['A1','Q2','K2','A2']).
    """
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for linea in f:
            linea = linea.strip()
            if not linea or linea.startswith('#'):
                continue
            if re.match(r'^<[^>]+>\s*<!', linea):
                continue
            m = re.match(r'^<([^.>]+)\.([^,]+),(@\w+|&\w+)>\s+(.*)', linea)
            if not m:
                continue
            folio = m.group(1)
            line_num = m.group(2)
            line_type = m.group(3)
            texto = m.group(4).strip()
            # Clean
            texto = re.sub(r'\{[^}]*\}', '', texto)
            texto = re.sub(r'<[^>]*>', '', texto)
            texto = re.sub(r'-', '.', texto)
            texto = texto.strip()
            if not texto:
                continue
            words = []
            for palabra_raw in texto.split('.'):
                palabra_raw = palabra_raw.strip()
                if not palabra_raw:
                    continue
                glifos = re.findall(r'[A-Z][0-9a-z]', palabra_raw)
                if glifos:
                    words.append(glifos)
            line_id = f'{folio}.{line_num}'
            records.append((folio, line_id, line_type, words))
    return records

STA_PATH = 'corpus/voynich_sta.txt'
records = parse_sta_with_folios(STA_PATH)
print(f'Total records: {len(records)}')
print(f'Distinct folios: {len(set(r[0] for r in records))}')

# Also parse full corpus for frequency comparison
all_words_flat = []
for _, _, _, words in records:
    for w in words:
        all_words_flat.append(tuple(w))
GLOBAL_FREQ = Counter(all_words_flat)
print(f'Total words in corpus: {len(all_words_flat):,}')
print(f'Distinct word types: {len(GLOBAL_FREQ):,}')

Total records: 1326
Distinct folios: 227
Total words in corpus: 5,279
Distinct word types: 3,092


In [2]:
# ── Cell 2: Extract zodiac pages ───────────────────────────────────────

ZODIAC_FOLIOS = {
    'f70v2': 'Pisces',
    'f70v1': 'Aries_1',
    'f71r':  'Aries_2',
    'f71v':  'Taurus_1',
    'f72r1': 'Taurus_2',
    'f72r2': 'Gemini',
    'f72r3': 'Cancer',
    'f72v3': 'Leo',
    'f72v2': 'Virgo',
    'f72v1': 'Libra',
    'f73r':  'Scorpio',
    'f73v':  'Sagittarius',
}

# Merge Aries and Taurus halves for combined analysis
SIGN_FOLIOS = {
    'Pisces':      ['f70v2'],
    'Aries':       ['f70v1', 'f71r'],
    'Taurus':      ['f71v', 'f72r1'],
    'Gemini':      ['f72r2'],
    'Cancer':      ['f72r3'],
    'Leo':         ['f72v3'],
    'Virgo':       ['f72v2'],
    'Libra':       ['f72v1'],
    'Scorpio':     ['f73r'],
    'Sagittarius': ['f73v'],
}

# Extract data per folio
zodiac_data = {}
for folio, sign in ZODIAC_FOLIOS.items():
    labels = []  # words from Lz lines (nymph labels)
    circular = []  # words from Cc lines (circular text)
    for f, lid, ltype, words in records:
        if f != folio:
            continue
        if ltype in ('@Lz', '&Lz'):
            labels.extend(words)
        elif ltype == '@Cc':
            circular.extend(words)
    zodiac_data[folio] = {
        'sign': sign,
        'labels': labels,
        'circular': circular,
        'all_words': labels + circular,
    }

# Summary
print(f'{"Folio":<8} {"Sign":<14} {"Labels":>6} {"Circular":>8} {"Total":>6}')
print('-' * 48)
total_labels = 0
total_circ = 0
for folio in ZODIAC_FOLIOS:
    d = zodiac_data[folio]
    nl = len(d['labels'])
    nc = len(d['circular'])
    total_labels += nl
    total_circ += nc
    print(f'{folio:<8} {d["sign"]:<14} {nl:>6} {nc:>8} {nl+nc:>6}')
print('-' * 48)
print(f'{"TOTAL":<8} {"":<14} {total_labels:>6} {total_circ:>8} {total_labels+total_circ:>6}')

Folio    Sign           Labels Circular  Total
------------------------------------------------
f70v2    Pisces             34       94    128
f70v1    Aries_1            25       57     82
f71r     Aries_2            17       69     86
f71v     Taurus_1           21       78     99
f72r1    Taurus_2           21       78     99
f72r2    Gemini             30       80    110
f72r3    Cancer             39      115    154
f72v3    Leo                34       84    118
f72v2    Virgo              30       65     95
f72v1    Libra              30       66     96
f73r     Scorpio            30       61     91
f73v     Sagittarius        30       63     93
------------------------------------------------
TOTAL                      341      910   1251


In [3]:
# ── Cell 3: Word uniqueness analysis ───────────────────────────────────

# Build set of all zodiac words and all non-zodiac words
zodiac_folios_set = set(ZODIAC_FOLIOS.keys())
non_zodiac_words = Counter()
zodiac_words_by_folio = {}

for folio in ZODIAC_FOLIOS:
    zodiac_words_by_folio[folio] = Counter(tuple(w) for w in zodiac_data[folio]['all_words'])

for _, lid, ltype, words in records:
    folio = lid.split('.')[0]
    if folio not in zodiac_folios_set:
        for w in words:
            non_zodiac_words[tuple(w)] += 1

# Words exclusive to each zodiac folio (not in rest of manuscript)
print('=== Words EXCLUSIVE to each zodiac folio (not found elsewhere) ===\n')
exclusive_by_folio = {}
for folio in ZODIAC_FOLIOS:
    exclusive = []
    folio_words = zodiac_words_by_folio[folio]
    other_zodiac = Counter()
    for f2 in ZODIAC_FOLIOS:
        if f2 != folio:
            other_zodiac.update(zodiac_words_by_folio[f2])
    for word, count in folio_words.most_common():
        if word not in non_zodiac_words and word not in other_zodiac:
            exclusive.append((' '.join(word), count))
    exclusive_by_folio[folio] = exclusive
    sign = ZODIAC_FOLIOS[folio]
    if exclusive:
        print(f'{folio} ({sign}): {len(exclusive)} exclusive words')
        for w, c in exclusive[:5]:
            print(f'  {w} (×{c})')
    else:
        print(f'{folio} ({sign}): 0 exclusive words')
    print()

# Words exclusive to zodiac section (not in rest of manuscript)
all_zodiac_counter = Counter()
for folio in ZODIAC_FOLIOS:
    all_zodiac_counter.update(zodiac_words_by_folio[folio])

exclusive_zodiac = [(w, c) for w, c in all_zodiac_counter.most_common()
                    if w not in non_zodiac_words]
print(f'\n=== Words exclusive to ALL zodiac pages: {len(exclusive_zodiac)} ===')
for w, c in exclusive_zodiac[:15]:
    print(f'  {" ".join(w)} (×{c})')

=== Words EXCLUSIVE to each zodiac folio (not found elsewhere) ===

f70v2 (Pisces): 51 exclusive words
  A1 Q2 A3 B2 A3 B2 B4 (×1)
  A1 Q1 A3 C2 A2 (×1)
  A1 Q2 A3 C1 Z1 (×1)
  A1 Ba A2 (×1)
  A1 Q2 A1 Ba Ab (×1)

f70v1 (Aries_1): 31 exclusive words
  A1 Q2 A3 B2 K1 A2 (×1)
  Q2 A3 C1 (×1)
  A1 Q2 A3 Q1 A3 E1 B3 A3 E2 (×1)
  A1 A3 B2 K1 J1 B4 (×1)
  A1 Q2 K1 A1 B1 A3 B2 C2 (×1)

f71r (Aries_2): 40 exclusive words
  A1 Q1 B2 Ba A3 B3 (×1)
  A1 Q2 J1 A1 A3 B2 Ba A2 (×1)
  A1 Q2 J1 A1 B2 A3 C1 (×1)
  A1 Q1 J1 A1 A3 B2 A2 (×1)
  A1 Q2 A3 B2 J1 Q1 A2 (×1)

f71v (Taurus_1): 40 exclusive words
  K1 P2 A3 B2 A2 (×1)
  A1 Q2 K1 A1 Ba A2 (×1)
  A3 B2 T1 A2 (×1)
  A1 Q1 A3 C1 A3 G1 (×1)
  A1 P1 A3 B2 A3 C1 (×1)

f72r1 (Taurus_2): 42 exclusive words
  A1 L1 A1 B1 A3 B1 A2 (×1)
  K1 B1 A3 G3 B1 A3 F1 A2 (×1)
  A1 Aa G1 A3 C1 A3 C1 A2 (×1)
  A1 Qa A3 B2 A3 B3 (×1)
  A2 Q2 A3 B2 L1 Ba A2 (×1)

f72r2 (Gemini): 36 exclusive words
  Q2 J1 A2 (×2)
  A1 Q2 M1 A2 (×2)
  A1 P1 K1 B1 A3 B3 A2 (×1)
  A1 Q1 B2

In [4]:
# ── Cell 4: First words of circular text and first labels ─────────────
# These are prime candidates for zodiac sign names

print('=== First word of first circular text (Cc) per folio ===\n')
first_cc = {}
for folio in ZODIAC_FOLIOS:
    for f, lid, ltype, words in records:
        if f == folio and ltype == '@Cc' and words:
            first_cc[folio] = words[0]
            break
    word_str = ' '.join(first_cc.get(folio, ['?']))
    sign = ZODIAC_FOLIOS[folio]
    # Check global frequency
    freq = GLOBAL_FREQ.get(tuple(first_cc.get(folio, [])), 0)
    print(f'  {folio:<8} ({sign:<14}): {word_str:<25} (global freq: {freq})')

print('\n=== First label (Lz) per folio ===\n')
first_lz = {}
for folio in ZODIAC_FOLIOS:
    for f, lid, ltype, words in records:
        if f == folio and ltype == '@Lz' and words:
            first_lz[folio] = words[0]
            break
    word_str = ' '.join(first_lz.get(folio, ['?']))
    sign = ZODIAC_FOLIOS[folio]
    freq = GLOBAL_FREQ.get(tuple(first_lz.get(folio, [])), 0)
    print(f'  {folio:<8} ({sign:<14}): {word_str:<25} (global freq: {freq})')

# Check if first Cc words are unique across signs
print('\n=== Repetition check: first Cc words ===')
cc_words = Counter(tuple(v) for v in first_cc.values())
for w, c in cc_words.most_common():
    if c > 1:
        folios = [f for f, v in first_cc.items() if tuple(v) == w]
        print(f'  "{" ".join(w)}" appears in {c} folios: {folios}')

print('\n=== Repetition check: first Lz labels ===')
lz_words = Counter(tuple(v) for v in first_lz.values())
for w, c in lz_words.most_common():
    if c > 1:
        folios = [f for f, v in first_lz.items() if tuple(v) == w]
        print(f'  "{" ".join(w)}" appears in {c} folios: {folios}')

=== First word of first circular text (Cc) per folio ===

  f70v2    (Pisces        ): A1 Q1 K1 J1 A1            (global freq: 1)
  f70v1    (Aries_1       ): B1 A3 B2 A3 B2 A1 B1 A2   (global freq: 1)
  f71r     (Aries_2       ): A1 B2 Q1 K2 A1 B1 A2      (global freq: 1)
  f71v     (Taurus_1      ): A1 L1 J1 A1               (global freq: 1)
  f72r1    (Taurus_2      ): A1 Q2 K2 A1               (global freq: 2)
  f72r2    (Gemini        ): A1 Q2 J1 A1 C2            (global freq: 16)
  f72r3    (Cancer        ): A1 Q1 K2 A2               (global freq: 21)
  f72v3    (Leo           ): C2 A1 K2 C2               (global freq: 1)
  f72v2    (Virgo         ): A1 Q2 A1 C2 K1 C2         (global freq: 1)
  f72v1    (Libra         ): A1 Q2 K2 A1 C2            (global freq: 8)
  f73r     (Scorpio       ): A2 P1 A1 B2 K1 K2 A2      (global freq: 1)
  f73v     (Sagittarius   ): A1 Q2 A1 A3 C1            (global freq: 3)

=== First label (Lz) per folio ===

  f70v2    (Pisces        ): A1 Q2 A3 B

In [5]:
# ── Cell 5: Label characteristics per sign ─────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Labels are typically 1-word, but some have multiple words on one line
# Analyze label lengths (in glyphs) per sign
print('=== Label word lengths (glyphs) per sign ===\n')

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes_flat = axes.flatten()

for idx, (sign, folios) in enumerate(SIGN_FOLIOS.items()):
    label_lengths = []
    label_words = []
    for folio in folios:
        for w in zodiac_data[folio]['labels']:
            label_lengths.append(len(w))
            label_words.append(tuple(w))

    ax = axes_flat[idx]
    if label_lengths:
        ax.hist(label_lengths, bins=range(1, max(label_lengths)+2),
                align='left', color='steelblue', edgecolor='white')
    ax.set_title(f'{sign} (n={len(label_lengths)})')
    ax.set_xlabel('Glyphs')
    ax.set_ylabel('Count')

    # Print most common labels
    lbl_freq = Counter(label_words)
    most_common = lbl_freq.most_common(3)
    mc_str = ', '.join(f'{" ".join(w)}(×{c})' for w, c in most_common)
    print(f'{sign:<14}: {len(label_lengths):>3} labels, '
          f'mean len={np.mean(label_lengths):.1f}, '
          f'top: {mc_str}')

# Hide unused subplots
for idx in range(len(SIGN_FOLIOS), len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('Label Word Lengths by Zodiac Sign', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/zodiac_label_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved: images/zodiac_label_lengths.png')

=== Label word lengths (glyphs) per sign ===

Pisces        :  34 labels, mean len=5.0, top: A1 Q2 A2(×2), A1 Q2 A3 B2 A3 B2 B4(×1), A1 Q1 A3 C2 A2(×1)
Aries         :  42 labels, mean len=5.7, top: A1 Q2 A3 B2 K1 A2(×1), Q2 A3 C1(×1), A3 B3(×1)
Taurus        :  42 labels, mean len=5.0, top: A1 Q2 A3 G1(×2), A1 Q2 A3 C1(×2), K1 A3 C1(×1)
Gemini        :  30 labels, mean len=5.4, top: A1 Q1 A3 B2(×2), A1 Q1 A3 B2 A2(×2), A1 P1 K1 B1 A3 B3 A2(×1)
Cancer        :  39 labels, mean len=5.3, top: A1 B2 Q1 A3 B2 A3 G1(×1), A1 B2 A3 B2 C2 A2(×1), A1 C1 A3 G1 A3 B3(×1)
Leo           :  34 labels, mean len=4.7, top: A1 K1 A1 B4(×1), A1 C1 M1 A2(×1), A2 Q1 A3 B3(×1)
Virgo         :  30 labels, mean len=5.6, top: K1 J1 A1 B2 B1 A2(×2), A1 K2 Ba J1 A2(×1), A1 K2 A1 Ba A3 G1(×1)
Libra         :  30 labels, mean len=4.7, top: A1 K2 A1 Q2 A2(×1), A1 U2 A2(×1), A1 Q2 J1 A1 B2 A2(×1)
Scorpio       :  30 labels, mean len=4.8, top: A1 Q2 A3 B2 A2(×2), K1 A1 U1 A2(×1), A1 Q1 J1 Ba A2(×1)
Sagittarius   :  3

/tmp/ipykernel_70529/2425731141.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── Cell 6: Hebrew zodiac names and crib compatibility ────────────────

# Hebrew zodiac names (traditional)
HEBREW_ZODIAC = {
    'Pisces':      list('דגים'),      # dagim (fish)
    'Aries':       list('טלה'),       # taleh (lamb)
    'Taurus':      list('שור'),       # shor (bull)
    'Gemini':      list('תאומים'),    # te'omim (twins)
    'Cancer':      list('סרטן'),      # sartan (crab)
    'Leo':         list('אריה'),      # aryeh (lion)
    'Virgo':       list('בתולה'),     # betulah (virgin)
    'Libra':       list('מאזנים'),    # moznayim (scales)
    'Scorpio':     list('עקרב'),      # akrav (scorpion)
    'Sagittarius': list('קשת'),       # keshet (bow)
}

# Alternative names (Aramaic/medieval variants)
ARAMAIC_ZODIAC = {
    'Pisces':      list('נוני'),      # nunei (fish, Aramaic)
    'Aries':       list('אמרא'),     # imra (lamb, Aramaic)
    'Taurus':      list('תורא'),     # tora (bull, Aramaic)
    'Gemini':      list('תאומי'),    # t'omei (Aramaic)
    'Cancer':      list('סרטנא'),    # sartana (Aramaic)
    'Leo':         list('אריא'),     # arya (Aramaic)
    'Virgo':       list('בתולתא'),   # btulta (Aramaic)
    'Libra':       list('מאזניא'),   # moznaiya (Aramaic)
    'Scorpio':     list('עקרבא'),    # akraba (Aramaic)
    'Sagittarius': list('קשתא'),     # kashta (Aramaic)
}

print('Hebrew zodiac names:')
for sign, chars in HEBREW_ZODIAC.items():
    print(f'  {sign:<14}: {"".join(chars)} ({len(chars)} chars)')

print('\nAramaic zodiac names:')
for sign, chars in ARAMAIC_ZODIAC.items():
    print(f'  {sign:<14}: {"".join(chars)} ({len(chars)} chars)')

# Key observation: sign names have 3-6 characters
# Voynich labels typically have 3-8 glyphs
# Under SilPart model (~2 glyphs per character), we'd expect 6-12 glyphs per name
# But labels average ~4-6 glyphs → could match 2-3 char names (שור, טלה, קשת)

print('\n=== Length compatibility ===')
print(f'{"Sign":<14} {"Heb chars":>9} {"Labels mean gl":>14} {"Ratio gl/ch":>11}')
for sign, folios in SIGN_FOLIOS.items():
    heb_len = len(HEBREW_ZODIAC[sign])
    label_lengths = []
    for folio in folios:
        for w in zodiac_data[folio]['labels']:
            label_lengths.append(len(w))
    mean_gl = np.mean(label_lengths) if label_lengths else 0
    ratio = mean_gl / heb_len if heb_len > 0 else 0
    print(f'{sign:<14} {heb_len:>9} {mean_gl:>14.1f} {ratio:>11.2f}')

Hebrew zodiac names:
  Pisces        : דגים (4 chars)
  Aries         : טלה (3 chars)
  Taurus        : שור (3 chars)
  Gemini        : תאומים (6 chars)
  Cancer        : סרטן (4 chars)
  Leo           : אריה (4 chars)
  Virgo         : בתולה (5 chars)
  Libra         : מאזנים (6 chars)
  Scorpio       : עקרב (4 chars)
  Sagittarius   : קשת (3 chars)

Aramaic zodiac names:
  Pisces        : נוני (4 chars)
  Aries         : אמרא (4 chars)
  Taurus        : תורא (4 chars)
  Gemini        : תאומי (5 chars)
  Cancer        : סרטנא (5 chars)
  Leo           : אריא (4 chars)
  Virgo         : בתולתא (6 chars)
  Libra         : מאזניא (6 chars)
  Scorpio       : עקרבא (5 chars)
  Sagittarius   : קשתא (4 chars)

=== Length compatibility ===
Sign           Heb chars Labels mean gl Ratio gl/ch
Pisces                 4            5.0        1.25
Aries                  3            5.7        1.90
Taurus                 3            5.0        1.67
Gemini                 6            5.4        0.

In [7]:
# ── Cell 7: Crib matching — test if candidate words can map to zodiac names ─

def check_mapping_consistency(glyph_word, char_word):
    """Check if a glyph sequence can map to a character sequence with a consistent
    1-glyph → 1-char mapping. Returns the mapping dict if consistent, None otherwise.
    """
    if len(glyph_word) != len(char_word):
        return None
    mapping = {}
    reverse = {}
    for g, c in zip(glyph_word, char_word):
        if g in mapping:
            if mapping[g] != c:
                return None  # glyph maps to two different chars
        else:
            mapping[g] = c
        if c in reverse:
            if reverse[c] != g:
                pass  # allow many-to-one (homophonic)
        reverse[c] = g
    return mapping


def check_mapping_consistency_many_to_one(glyph_word, char_word):
    """Like above but explicitly allows many-to-one (multiple glyphs → same char).
    Only rejects if ONE glyph maps to TWO different chars.
    """
    if len(glyph_word) != len(char_word):
        return None
    mapping = {}
    for g, c in zip(glyph_word, char_word):
        if g in mapping:
            if mapping[g] != c:
                return None
        else:
            mapping[g] = c
    return mapping


# For each sign, collect candidate words (labels with right length range)
print('=== Crib candidates: labels with compatible length ===\n')

crib_candidates = {}
for sign, folios in SIGN_FOLIOS.items():
    heb_name = HEBREW_ZODIAC[sign]
    ara_name = ARAMAIC_ZODIAC[sign]
    heb_len = len(heb_name)
    ara_len = len(ara_name)

    candidates = []
    for folio in folios:
        # Check all labels
        for w in zodiac_data[folio]['labels']:
            # Try Hebrew name
            m = check_mapping_consistency_many_to_one(w, heb_name)
            if m is not None:
                candidates.append(('hebrew', tuple(w), ''.join(heb_name), m, folio))
            # Try Aramaic name
            m = check_mapping_consistency_many_to_one(w, ara_name)
            if m is not None:
                candidates.append(('aramaic', tuple(w), ''.join(ara_name), m, folio))
        # Also check first word of circular text
        for f, lid, ltype, words in records:
            if f == folio and ltype == '@Cc' and words:
                w = words[0]
                m = check_mapping_consistency_many_to_one(w, heb_name)
                if m is not None:
                    candidates.append(('hebrew-Cc', tuple(w), ''.join(heb_name), m, folio))
                m = check_mapping_consistency_many_to_one(w, ara_name)
                if m is not None:
                    candidates.append(('aramaic-Cc', tuple(w), ''.join(ara_name), m, folio))
                break  # only first Cc

    crib_candidates[sign] = candidates
    if candidates:
        print(f'{sign} ({heb_len}ch / {ara_len}ch):')
        for lang, gw, cn, mp, fol in candidates:
            gw_str = ' '.join(gw)
            mp_str = ', '.join(f'{g}→{c}' for g, c in mp.items())
            print(f'  [{lang}] {gw_str} → {cn}  ({mp_str})  [from {fol}]')
    else:
        print(f'{sign}: no length-compatible candidates')
    print()

total = sum(len(v) for v in crib_candidates.values())
print(f'Total crib candidates: {total}')

=== Crib candidates: labels with compatible length ===

Pisces (4ch / 4ch):
  [hebrew] A1 Q2 A3 B2 → דגים  (A1→ד, Q2→ג, A3→י, B2→ם)  [from f70v2]
  [aramaic] A1 Q2 A3 B2 → נוני  (A1→נ, Q2→ו, A3→נ, B2→י)  [from f70v2]
  [hebrew] A1 Q1 K2 C2 → דגים  (A1→ד, Q1→ג, K2→י, C2→ם)  [from f70v2]
  [aramaic] A1 Q1 K2 C2 → נוני  (A1→נ, Q1→ו, K2→נ, C2→י)  [from f70v2]
  [aramaic] A3 Q2 A3 C1 → נוני  (A3→נ, Q2→ו, C1→י)  [from f70v2]

Aries (3ch / 4ch):
  [hebrew] Q2 A3 C1 → טלה  (Q2→ט, A3→ל, C1→ה)  [from f70v1]
  [hebrew] C2 A3 B2 → טלה  (C2→ט, A3→ל, B2→ה)  [from f70v1]
  [aramaic] A1 Q2 Aa B2 → אמרא  (A1→א, Q2→מ, Aa→ר, B2→א)  [from f70v1]
  [aramaic] A1 Q2 J1 A1 → אמרא  (A1→א, Q2→מ, J1→ר)  [from f70v1]
  [aramaic] A1 Q2 A3 B2 → אמרא  (A1→א, Q2→מ, A3→ר, B2→א)  [from f70v1]
  [aramaic] C2 A3 C1 A2 → אמרא  (C2→א, A3→מ, C1→ר, A2→א)  [from f71r]

Taurus (3ch / 4ch):
  [hebrew] K1 A3 C1 → שור  (K1→ש, A3→ו, C1→ר)  [from f71v]
  [aramaic] A3 B2 T1 A2 → תורא  (A3→ת, B2→ו, T1→ר, A2→א)  [from f71v]
  [aramaic

In [8]:
# ── Cell 8: Cross-sign mapping consistency (greedy, no combinatorial explosion) ─
# Instead of enumerating all combinations (billions), we use a greedy approach:
# 1. Check pairwise consistency between all sign pairs
# 2. Build a global mapping greedily, adding signs one by one

# Collect all per-sign candidate mappings
sign_maps = {}
for sign, candidates in crib_candidates.items():
    if candidates:
        sign_maps[sign] = [(c[1], c[2], c[3], c[0]) for c in candidates]

if len(sign_maps) < 2:
    print('Not enough signs with candidates to check cross-consistency.')
    best_global = None
    best_merged = None
else:
    signs_with_cands = sorted(sign_maps.keys())
    print(f'Signs with candidates: {len(signs_with_cands)}')
    cands_per_sign = [len(sign_maps[s]) for s in signs_with_cands]
    print(f'Candidates per sign: {dict(zip(signs_with_cands, cands_per_sign))}')
    
    total_combos = 1
    for c in cands_per_sign:
        total_combos *= c
    print(f'Full cartesian product: {total_combos:,.0f} (too large for brute force)')
    
    # ── Step 1: Pairwise consistency matrix ──
    print(f'\n=== Pairwise consistency ===\n')
    for i, s1 in enumerate(signs_with_cands):
        for s2 in signs_with_cands[i+1:]:
            n_consistent = 0
            n_total = 0
            best_overlap = 0
            best_pair = None
            for c1 in sign_maps[s1]:
                for c2 in sign_maps[s2]:
                    n_total += 1
                    m1, m2 = c1[2], c2[2]
                    shared = set(m1.keys()) & set(m2.keys())
                    consistent = all(m1[g] == m2[g] for g in shared)
                    if consistent:
                        n_consistent += 1
                        if len(shared) > best_overlap:
                            best_overlap = len(shared)
                            best_pair = (c1, c2)
            
            if best_overlap > 0:
                print(f'  {s1} × {s2}: {n_consistent}/{n_total} consistent, '
                      f'best overlap: {best_overlap} shared glyphs')
                c1, c2 = best_pair
                shared = set(c1[2].keys()) & set(c2[2].keys())
                for g in shared:
                    print(f'    {g} → {c1[2][g]} (confirmed by both signs)')
            elif n_consistent > 0:
                print(f'  {s1} × {s2}: {n_consistent}/{n_total} consistent (no shared glyphs)')
            else:
                print(f'  {s1} × {s2}: ALL contradictory')
    
    # ── Step 2: Greedy global assignment ──
    print(f'\n=== Greedy global consistency search ===\n')
    
    # Try each sign as starting point, greedily add compatible candidates
    best_global = None
    best_merged = None
    best_n_signs = 0
    
    for start_sign in signs_with_cands:
        for start_cand in sign_maps[start_sign]:
            # Start with this candidate
            current_mapping = dict(start_cand[2])
            selected = {start_sign: start_cand}
            
            # Try to add other signs in order of fewest candidates (most constrained first)
            remaining = sorted(
                [s for s in signs_with_cands if s != start_sign],
                key=lambda s: len(sign_maps[s])
            )
            
            for next_sign in remaining:
                # Find best compatible candidate for this sign
                best_cand = None
                best_shared = -1
                for cand in sign_maps[next_sign]:
                    m = cand[2]
                    shared = set(m.keys()) & set(current_mapping.keys())
                    consistent = all(m[g] == current_mapping[g] for g in shared)
                    if consistent and len(shared) > best_shared:
                        best_shared = len(shared)
                        best_cand = cand
                
                if best_cand is not None:
                    selected[next_sign] = best_cand
                    current_mapping.update(best_cand[2])
            
            if len(selected) > best_n_signs or (
                len(selected) == best_n_signs and 
                len(current_mapping) > len(best_merged or {})
            ):
                best_n_signs = len(selected)
                best_merged = dict(current_mapping)
                best_global = dict(selected)
    
    if best_global:
        print(f'Best globally consistent set: {best_n_signs} signs')
        print(f'Merged mapping ({len(best_merged)} glyph→char pairs):\n')
        for sign in signs_with_cands:
            if sign in best_global:
                entry = best_global[sign]
                print(f'  {sign:<14}: {" ".join(entry[0])} → {entry[1]} [{entry[3]}]')
            else:
                print(f'  {sign:<14}: (excluded — contradicts)')
        print(f'\nGlyph → Hebrew char mapping:')
        for g, c in sorted(best_merged.items()):
            print(f'  {g} → {c}')
        
        # Check for contradictions in the mapping
        n_glyphs_used = len(best_merged)
        n_chars_used = len(set(best_merged.values()))
        print(f'\n{n_glyphs_used} glyphs → {n_chars_used} distinct Hebrew chars')
    else:
        print('No globally consistent assignment found.')

Signs with candidates: 9
Candidates per sign: {'Aries': 6, 'Cancer': 12, 'Gemini': 10, 'Leo': 19, 'Pisces': 5, 'Sagittarius': 11, 'Scorpio': 12, 'Taurus': 13, 'Virgo': 10}
Full cartesian product: 1,173,744,000 (too large for brute force)

=== Pairwise consistency ===

  Aries × Cancer: 8/72 consistent (no shared glyphs)
  Aries × Gemini: 1/60 consistent (no shared glyphs)
  Aries × Leo: 60/114 consistent, best overlap: 2 shared glyphs
    A1 → א (confirmed by both signs)
    B2 → א (confirmed by both signs)
  Aries × Pisces: 2/30 consistent (no shared glyphs)
  Aries × Sagittarius: 20/66 consistent, best overlap: 1 shared glyphs
    A2 → א (confirmed by both signs)
  Aries × Scorpio: 16/72 consistent, best overlap: 1 shared glyphs
    B2 → א (confirmed by both signs)
  Aries × Taurus: 19/78 consistent, best overlap: 1 shared glyphs
    C1 → ר (confirmed by both signs)
  Aries × Virgo: 12/60 consistent, best overlap: 2 shared glyphs
    A3 → ל (confirmed by both signs)
    C1 → ה (confi

In [9]:
# ── Cell 9: Statistical significance — is 4 consistent signs by chance? ────
# Monte Carlo null: randomly assign zodiac names to labels, check consistency

N_NULL = 10_000
rng_null = np.random.default_rng(42)

# Gather the actual candidate pools per sign (word tuples + their lengths)
actual_labels = {}
for sign, folios in SIGN_FOLIOS.items():
    labs = []
    for folio in folios:
        labs.extend([tuple(w) for w in zodiac_data[folio]['labels']])
    actual_labels[sign] = labs

# The zodiac names we test against (both Hebrew and Aramaic)
zodiac_names_all = {}
for sign in SIGN_FOLIOS:
    zodiac_names_all[sign] = []
    for name_set, label in [(HEBREW_ZODIAC, 'heb'), (ARAMAIC_ZODIAC, 'ara')]:
        zodiac_names_all[sign].append((name_set[sign], label))

signs_list = sorted(SIGN_FOLIOS.keys())

def find_candidates_for_sign(sign, labels, names):
    """Find all length-compatible, internally consistent label→name mappings."""
    cands = []
    for name_chars, lang in names:
        for lab in labels:
            if len(lab) != len(name_chars):
                continue
            mapping = {}
            ok = True
            for g, c in zip(lab, name_chars):
                if g in mapping:
                    if mapping[g] != c:
                        ok = False
                        break
                else:
                    mapping[g] = c
            if ok:
                cands.append(mapping)
    return cands

def greedy_max_consistent(sign_cands_dict, signs):
    """Greedy search: return max number of mutually consistent signs."""
    best_n = 0
    for start_sign in signs:
        if start_sign not in sign_cands_dict or not sign_cands_dict[start_sign]:
            continue
        for start_map in sign_cands_dict[start_sign]:
            merged = dict(start_map)
            count = 1
            for next_sign in signs:
                if next_sign == start_sign or next_sign not in sign_cands_dict:
                    continue
                found = False
                for cand_map in sign_cands_dict[next_sign]:
                    shared = set(cand_map.keys()) & set(merged.keys())
                    if all(cand_map[g] == merged[g] for g in shared):
                        merged.update(cand_map)
                        count += 1
                        found = True
                        break
                # If no compatible candidate, skip this sign
            if count > best_n:
                best_n = count
    return best_n

# Observed value
observed_n_signs = best_n_signs  # 4 from cell 8
print(f'Observed: {observed_n_signs} mutually consistent signs')
print(f'Running {N_NULL:,} null simulations...\n')

null_counts = []
for sim in range(N_NULL):
    # Shuffle: randomly assign zodiac names to signs
    shuffled_names = dict(zip(signs_list, rng_null.permutation(signs_list)))
    
    sign_cands = {}
    for sign in signs_list:
        # Use the LABELS from this sign's actual page
        labs = actual_labels[sign]
        # But try to match them against a RANDOM sign's zodiac name
        random_sign = shuffled_names[sign]
        names = zodiac_names_all[random_sign]
        cands = find_candidates_for_sign(sign, labs, names)
        if cands:
            sign_cands[sign] = cands
    
    n = greedy_max_consistent(sign_cands, signs_list)
    null_counts.append(n)

null_counts = np.array(null_counts)
p_value = np.mean(null_counts >= observed_n_signs)
mean_null = np.mean(null_counts)
max_null = np.max(null_counts)

print(f'Null distribution: mean={mean_null:.2f}, max={max_null}')
print(f'P(≥{observed_n_signs} signs | null) = {p_value:.4f}')
print(f'  → {"SIGNIFICANT (p<0.05)" if p_value < 0.05 else "NOT significant"}')

# Histogram
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
bins = range(0, max_null + 3)
ax.hist(null_counts, bins=bins, align='left', color='steelblue', edgecolor='white',
        density=True, label='Null distribution')
ax.axvline(observed_n_signs, color='red', linestyle='--', linewidth=2,
           label=f'Observed = {observed_n_signs} (p={p_value:.3f})')
ax.set_xlabel('Max consistent signs')
ax.set_ylabel('Density')
ax.set_title('Zodiac Crib Consistency: Observed vs Null')
ax.legend()
plt.tight_layout()
plt.savefig('images/zodiac_crib_null.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/zodiac_crib_null.png')

Observed: 4 mutually consistent signs
Running 10,000 null simulations...

Null distribution: mean=3.62, max=5
P(≥4 signs | null) = 0.6103
  → NOT significant
Saved: images/zodiac_crib_null.png


/tmp/ipykernel_70529/3524483411.py:119: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ── Cell 10: Validate crib mapping against full corpus ────────────────
# Apply the 14-glyph mapping to the entire Voynich corpus and check:
# 1. Coverage: what fraction of glyphs are mapped?
# 2. N-gram validity: do decoded bigrams match Hebrew?
# 3. C/V role consistency: do vowel/consonant roles match?

if best_merged is None:
    print('No crib mapping to validate.')
else:
    # ── Full corpus words ──
    all_corpus_words = []
    for _, _, _, words in records:
        for w in words:
            all_corpus_words.append(w)
    
    # ── Coverage ──
    all_glyphs = [g for w in all_corpus_words for g in w]
    mapped_glyphs = sum(1 for g in all_glyphs if g in best_merged)
    coverage = mapped_glyphs / len(all_glyphs) * 100
    print(f'=== Coverage ===')
    print(f'Mapped glyphs: {len(best_merged)} types, covering {mapped_glyphs:,}/{len(all_glyphs):,} tokens = {coverage:.1f}%\n')
    
    # ── Partially decode corpus ──
    decoded_pairs = []  # (glyph_bigram, hebrew_char) for consecutive mapped glyphs
    for w in all_corpus_words:
        for i in range(len(w) - 1):
            g1, g2 = w[i], w[i+1]
            if g1 in best_merged and g2 in best_merged:
                decoded_pairs.append((best_merged[g1], best_merged[g2]))
    
    # ── Hebrew bigram reference (from Torah) ──
    import requests
    
    def download_torah_chars():
        libros = ['Genesis', 'Exodus', 'Leviticus', 'Numbers', 'Deuteronomy']
        base = ('https://raw.githubusercontent.com/Sefaria/Sefaria-Export/'
                'master/json/Tanakh/Torah/{}/Hebrew/Tanach%20with%20Text%20Only.json')
        words = []
        for libro in libros:
            resp = requests.get(base.format(libro), timeout=30)
            resp.raise_for_status()
            data = resp.json()
            import re as re2
            for cap in data['text']:
                for vers in cap:
                    texto = re2.sub(r'[\u0591-\u05C7]', '', vers)
                    pals = re2.findall(r'[\u05D0-\u05EA]+', texto)
                    for p in pals:
                        if 2 <= len(p) <= 15:
                            words.append(list(p))
        return words
    
    print('Downloading Torah for bigram reference...')
    torah_words = download_torah_chars()
    
    # Hebrew bigrams
    heb_bigrams = Counter()
    for w in torah_words:
        for i in range(len(w) - 1):
            heb_bigrams[(w[i], w[i+1])] += 1
    
    # Check decoded bigrams
    n_valid = sum(1 for pair in decoded_pairs if pair in heb_bigrams)
    n_total = len(decoded_pairs)
    bigram_rate = n_valid / max(n_total, 1) * 100
    
    print(f'\n=== Bigram validity ===')
    print(f'Consecutive mapped glyph pairs: {n_total}')
    print(f'Valid Hebrew bigrams: {n_valid}/{n_total} = {bigram_rate:.1f}%')
    
    # Null: random mapping of same 14 glyphs
    heb_chars = sorted(set(c for w in torah_words for c in w))
    rng_bg = np.random.default_rng(42)
    null_rates = []
    for _ in range(1000):
        null_map = {g: rng_bg.choice(heb_chars) for g in best_merged}
        null_pairs = []
        for w in all_corpus_words:
            for i in range(len(w) - 1):
                if w[i] in null_map and w[i+1] in null_map:
                    null_pairs.append((null_map[w[i]], null_map[w[i+1]]))
        null_valid = sum(1 for p in null_pairs if p in heb_bigrams)
        null_rates.append(null_valid / max(len(null_pairs), 1) * 100)
    
    null_mean = np.mean(null_rates)
    null_std = np.std(null_rates)
    z_bigram = (bigram_rate - null_mean) / max(null_std, 1e-10)
    pct_bigram = np.mean([1 for x in null_rates if bigram_rate > x]) / len(null_rates) * 100
    
    print(f'Null bigram rate: {null_mean:.1f}% ± {null_std:.1f}%')
    print(f'Z-score: {z_bigram:.2f}, percentile: {pct_bigram:.1f}%')
    
    # ── C/V role consistency ──
    # Hebrew matres lectionis (vowel letters): א ה ו י
    HEBREW_VOWELS = set('אהוי')
    HEBREW_CONSONANTS = set('בגדזחטכלמנסעפצקרשת')
    
    print(f'\n=== C/V role check ===')
    print(f'{"Glyph":<6} {"→ Heb":>5} {"Role":>8}')
    n_vowel = 0
    n_consonant = 0
    for g, c in sorted(best_merged.items()):
        if c in HEBREW_VOWELS:
            role = 'VOWEL'
            n_vowel += 1
        elif c in HEBREW_CONSONANTS:
            role = 'CONS'
            n_consonant += 1
        else:
            role = '?'
        print(f'  {g:<6} → {c:>3}   {role}')
    print(f'\n{n_vowel} map to vowels, {n_consonant} map to consonants')
    print(f'Vowel fraction: {n_vowel}/{n_vowel+n_consonant} = {n_vowel/(n_vowel+n_consonant)*100:.0f}%')
    print(f'(Hebrew has 4/27 = 15% vowel chars; higher here suggests homophonic collapse onto vowels)')
    
    # ── Decode sample: zodiac labels ──
    print(f'\n=== Sample decoded zodiac labels ===')
    for folio in list(ZODIAC_FOLIOS.keys())[:4]:
        sign = ZODIAC_FOLIOS[folio]
        print(f'\n--- {folio} ({sign}) ---')
        for w in zodiac_data[folio]['labels'][:8]:
            decoded = ''.join(best_merged.get(g, '·') for g in w)
            w_str = ' '.join(w)
            print(f'  {w_str:<30} → {decoded}')

=== Coverage ===
Mapped glyphs: 14 types, covering 18,338/24,443 tokens = 75.0%


=== Bigram validity ===
Consecutive mapped glyph pairs: 9582
Valid Hebrew bigrams: 9581/9582 = 100.0%
Null bigram rate: 75.3% ± 16.1%
Z-score: 1.54, percentile: 0.1%

=== C/V role check ===
Glyph  → Heb     Role
  A1     →   ו   VOWEL
  A2     →   א   VOWEL
  A3     →   ב   CONS
  Aa     →   ו   VOWEL
  B2     →   א   VOWEL
  B3     →   ה   VOWEL
  C1     →   ר   CONS
  C2     →   ת   CONS
  E1     →   ל   CONS
  F3     →   ת   CONS
  J1     →   ק   CONS
  K1     →   ש   CONS
  L1     →   ע   CONS
  Q1     →   ר   CONS

5 map to vowels, 9 map to consonants
Vowel fraction: 5/14 = 36%
(Hebrew has 4/27 = 15% vowel chars; higher here suggests homophonic collapse onto vowels)

=== Sample decoded zodiac labels ===

--- f70v2 (Pisces) ---
  A1 Q2 A3 B2 A3 B2 B4           → ו·באבא·
  A1 Q1 A3 C2 A2                 → ורבתא
  A1 Q2 A3 C1 Z1                 → ו·בר·
  A1 Q2 A2                       → ו·א
  A1 Q1 A2  

In [11]:
# ── Cell 10b: Search circular text (Cc) for zodiac names ──────────────
# Strategy: the zodiac sign name is more likely in the circular text than
# in the degree-labels. We search ALL Cc words (not just the first one)
# and apply stricter filtering:
#   1. Length-compatible with Hebrew or Aramaic zodiac name
#   2. Internally consistent glyph→char mapping
#   3. Prefer words UNIQUE to that zodiac page (or rare elsewhere)

print('=== Crib search in circular text (Cc) ===\n')

# Collect Cc words per sign with global frequency info
cc_candidates = {}
for sign, folios in SIGN_FOLIOS.items():
    heb_name = HEBREW_ZODIAC[sign]
    ara_name = ARAMAIC_ZODIAC[sign]
    
    candidates = []
    for folio in folios:
        cc_words = zodiac_data[folio]['circular']
        for w_idx, w in enumerate(cc_words):
            for name, lang in [(heb_name, 'hebrew'), (ara_name, 'aramaic')]:
                m = check_mapping_consistency_many_to_one(w, name)
                if m is not None:
                    freq = GLOBAL_FREQ.get(tuple(w), 0)
                    candidates.append({
                        'lang': lang,
                        'word': tuple(w),
                        'target': ''.join(name),
                        'mapping': m,
                        'folio': folio,
                        'position': w_idx,
                        'global_freq': freq,
                    })
    
    # Sort by rarity (lower global frequency = more likely to be the sign name)
    candidates.sort(key=lambda x: x['global_freq'])
    cc_candidates[sign] = candidates
    
    n_heb = sum(1 for c in candidates if c['lang'] == 'hebrew')
    n_ara = sum(1 for c in candidates if c['lang'] == 'aramaic')
    n_rare = sum(1 for c in candidates if c['global_freq'] <= 2)
    print(f'{sign:<14}: {len(candidates):>3} Cc candidates '
          f'({n_heb} heb, {n_ara} ara, {n_rare} rare[freq≤2])')
    # Show top 3 rarest
    for c in candidates[:3]:
        w_str = ' '.join(c['word'])
        mp_str = ', '.join(f'{g}→{c_}' for g, c_ in c['mapping'].items())
        print(f'    [{c["lang"]}] {w_str} → {c["target"]}  '
              f'(freq={c["global_freq"]}, pos={c["position"]})  ({mp_str})')
    print()

# ── Combine labels + Cc candidates ──
print('=== Combined candidates (labels + circular text) ===\n')
combined_candidates = {}
for sign in SIGN_FOLIOS:
    combined = []
    
    # Add label candidates from cell 7
    for c in crib_candidates.get(sign, []):
        combined.append({
            'lang': c[0], 'word': c[1], 'target': c[2],
            'mapping': c[3], 'folio': c[4],
            'source': 'label',
            'global_freq': GLOBAL_FREQ.get(c[1], 0),
        })
    
    # Add Cc candidates
    for c in cc_candidates.get(sign, []):
        combined.append({
            'lang': c['lang'], 'word': c['word'], 'target': c['target'],
            'mapping': c['mapping'], 'folio': c['folio'],
            'source': 'circular',
            'global_freq': c['global_freq'],
        })
    
    combined_candidates[sign] = combined
    print(f'{sign:<14}: {len(combined):>4} total '
          f'({sum(1 for c in combined if c["source"]=="label")} label + '
          f'{sum(1 for c in combined if c["source"]=="circular")} Cc)')

total_combined = sum(len(v) for v in combined_candidates.values())
print(f'\nTotal combined candidates: {total_combined}')

=== Crib search in circular text (Cc) ===

Pisces        :  40 Cc candidates (19 heb, 21 ara, 10 rare[freq≤2])
    [aramaic] A3 B2 A3 F1 → נוני  (freq=1, pos=17)  (A3→נ, B2→ו, F1→י)
    [hebrew] L1 K1 J1 A2 → דגים  (freq=1, pos=50)  (L1→ד, K1→ג, J1→י, A2→ם)
    [aramaic] L1 K1 J1 A2 → נוני  (freq=1, pos=50)  (L1→נ, K1→ו, J1→נ, A2→י)

Aries         :  40 Cc candidates (25 heb, 15 ara, 9 rare[freq≤2])
    [hebrew] Cd A3 G1 → טלה  (freq=1, pos=4)  (Cd→ט, A3→ל, G1→ה)
    [hebrew] A1 K1 A2 → טלה  (freq=1, pos=6)  (A1→ט, K1→ל, A2→ה)
    [hebrew] B3 A3 C1 → טלה  (freq=1, pos=22)  (B3→ט, A3→ל, C1→ה)

Taurus        :  73 Cc candidates (29 heb, 44 ara, 21 rare[freq≤2])
    [aramaic] A3 Q2 K2 A2 → תורא  (freq=1, pos=4)  (A3→ת, Q2→ו, K2→ר, A2→א)
    [hebrew] L1 K2 A1 → שור  (freq=1, pos=14)  (L1→ש, K2→ו, A1→ר)
    [aramaic] Z1 K1 J1 A2 → תורא  (freq=1, pos=16)  (Z1→ת, K1→ו, J1→ר, A2→א)

Gemini        :  14 Cc candidates (5 heb, 9 ara, 9 rare[freq≤2])
    [hebrew] Q1 J1 A1 B1 A3 B2 → תאומים  (freq=

In [12]:
# ── Cell 10c: Greedy consistency + Monte Carlo on combined candidates ──
# Now using RARE candidates only (freq ≤ 3) to reduce noise

FREQ_THRESHOLD = 3  # only consider words appearing ≤3 times in corpus

# Build sign_maps with rare combined candidates
sign_maps_v2 = {}
for sign, cands in combined_candidates.items():
    rare = [c for c in cands if c['global_freq'] <= FREQ_THRESHOLD]
    if rare:
        sign_maps_v2[sign] = [(c['word'], c['target'], c['mapping'], 
                                f"{c['lang']}-{c['source']}") for c in rare]

signs_v2 = sorted(sign_maps_v2.keys())
print(f'Signs with rare candidates (freq≤{FREQ_THRESHOLD}): {len(signs_v2)}')
for s in signs_v2:
    print(f'  {s:<14}: {len(sign_maps_v2[s])} rare candidates')

# ── Greedy search ──
print(f'\n=== Greedy global consistency (rare candidates only) ===\n')

best_global_v2 = None
best_merged_v2 = None
best_n_v2 = 0

for start_sign in signs_v2:
    for start_cand in sign_maps_v2[start_sign]:
        current_mapping = dict(start_cand[2])
        selected = {start_sign: start_cand}
        
        remaining = sorted(
            [s for s in signs_v2 if s != start_sign],
            key=lambda s: len(sign_maps_v2[s])
        )
        
        for next_sign in remaining:
            best_cand = None
            best_shared = -1
            for cand in sign_maps_v2[next_sign]:
                m = cand[2]
                shared = set(m.keys()) & set(current_mapping.keys())
                consistent = all(m[g] == current_mapping[g] for g in shared)
                if consistent and len(shared) > best_shared:
                    best_shared = len(shared)
                    best_cand = cand
            
            if best_cand is not None:
                selected[next_sign] = best_cand
                current_mapping.update(best_cand[2])
        
        if len(selected) > best_n_v2 or (
            len(selected) == best_n_v2 and
            len(current_mapping) > len(best_merged_v2 or {})
        ):
            best_n_v2 = len(selected)
            best_merged_v2 = dict(current_mapping)
            best_global_v2 = dict(selected)

if best_global_v2:
    print(f'Best globally consistent set: {best_n_v2} signs')
    print(f'Merged mapping ({len(best_merged_v2)} glyph→char pairs):\n')
    for sign in signs_v2:
        if sign in best_global_v2:
            entry = best_global_v2[sign]
            src = entry[3]
            print(f'  {sign:<14}: {" ".join(entry[0])} → {entry[1]} [{src}] '
                  f'(freq={GLOBAL_FREQ.get(entry[0], 0)})')
        else:
            print(f'  {sign:<14}: (excluded)')
    print(f'\nGlyph → Hebrew char mapping:')
    for g, c in sorted(best_merged_v2.items()):
        print(f'  {g} → {c}')

# ── Monte Carlo null test ──
print(f'\n=== Monte Carlo null test (N=10,000) ===\n')

N_NULL_V2 = 10_000
rng_v2 = np.random.default_rng(123)

# Precompute: all Cc + label words per sign
all_sign_words = {}
for sign, folios in SIGN_FOLIOS.items():
    words = []
    for folio in folios:
        words.extend([tuple(w) for w in zodiac_data[folio]['labels']])
        words.extend([tuple(w) for w in zodiac_data[folio]['circular']])
    all_sign_words[sign] = words

null_counts_v2 = []
for sim in range(N_NULL_V2):
    shuffled_names = dict(zip(signs_list, rng_v2.permutation(signs_list)))
    
    sign_cands_null = {}
    for sign in signs_list:
        labs = all_sign_words[sign]
        random_sign = shuffled_names[sign]
        names = zodiac_names_all[random_sign]
        cands = []
        for name_chars, lang in names:
            for lab in labs:
                if len(lab) != len(name_chars):
                    continue
                # Freq filter
                if GLOBAL_FREQ.get(lab, 0) > FREQ_THRESHOLD:
                    continue
                mapping = {}
                ok = True
                for g, c in zip(lab, name_chars):
                    if g in mapping:
                        if mapping[g] != c:
                            ok = False
                            break
                    else:
                        mapping[g] = c
                if ok:
                    cands.append(mapping)
        if cands:
            sign_cands_null[sign] = cands
    
    n = greedy_max_consistent(sign_cands_null, signs_list)
    null_counts_v2.append(n)

null_counts_v2 = np.array(null_counts_v2)
observed_v2 = best_n_v2
p_value_v2 = np.mean(null_counts_v2 >= observed_v2)
mean_null_v2 = np.mean(null_counts_v2)
max_null_v2 = np.max(null_counts_v2)

print(f'Observed: {observed_v2} consistent signs (rare candidates)')
print(f'Null: mean={mean_null_v2:.2f}, max={max_null_v2}')
print(f'P(≥{observed_v2} | null) = {p_value_v2:.4f}')
print(f'  → {"SIGNIFICANT (p<0.05)" if p_value_v2 < 0.05 else "NOT significant"}')

# Histogram
fig, ax = plt.subplots(figsize=(8, 4))
bins = range(0, int(max_null_v2) + 3)
ax.hist(null_counts_v2, bins=bins, align='left', color='steelblue', 
        edgecolor='white', density=True, label='Null (shuffled names)')
ax.axvline(observed_v2, color='red', linestyle='--', linewidth=2,
           label=f'Observed = {observed_v2} (p={p_value_v2:.3f})')
ax.set_xlabel('Max consistent signs (rare cands, labels+Cc)')
ax.set_ylabel('Density')
ax.set_title(f'Zodiac Cribs: Labels + Circular Text (freq≤{FREQ_THRESHOLD})')
ax.legend()
plt.tight_layout()
plt.savefig('images/zodiac_crib_combined_null.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/zodiac_crib_combined_null.png')

Signs with rare candidates (freq≤3): 10
  Aries         : 15 rare candidates
  Cancer        : 34 rare candidates
  Gemini        : 20 rare candidates
  Leo           : 28 rare candidates
  Libra         : 6 rare candidates
  Pisces        : 15 rare candidates
  Sagittarius   : 16 rare candidates
  Scorpio       : 21 rare candidates
  Taurus        : 33 rare candidates
  Virgo         : 28 rare candidates

=== Greedy global consistency (rare candidates only) ===

Best globally consistent set: 5 signs
Merged mapping (17 glyph→char pairs):

  Aries         : A1 Q2 Aa B2 → אמרא [aramaic-label] (freq=3)
  Cancer        : (excluded)
  Gemini        : (excluded)
  Leo           : A1 C1 M1 A2 → אריה [hebrew-label] (freq=1)
  Libra         : (excluded)
  Pisces        : B1 A3 E1 B3 → דגים [hebrew-circular] (freq=3)
  Sagittarius   : Q1 K2 C2 → קשת [hebrew-circular] (freq=1)
  Scorpio       : (excluded)
  Taurus        : U2 J1 Ab → שור [hebrew-circular] (freq=1)
  Virgo         : (excluded)

Gl

/tmp/ipykernel_70529/749687480.py:147: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# ── Cell 10d: Exclusivity filter — only words unique to ONE sign ───────
# Key insight: if a word appears on Pisces AND Aries pages, it can't be
# a zodiac sign name. Only words EXCLUSIVE to one sign are valid candidates.

# Build word sets per SIGN (merging halves for Aries/Taurus)
words_per_sign = {}
for sign, folios in SIGN_FOLIOS.items():
    word_set = set()
    for folio in folios:
        for w in zodiac_data[folio]['all_words']:
            word_set.add(tuple(w))
    words_per_sign[sign] = word_set

# Find words exclusive to each sign (not in any OTHER zodiac sign's pages)
exclusive_per_sign = {}
for sign in SIGN_FOLIOS:
    other_words = set()
    for s2 in SIGN_FOLIOS:
        if s2 != sign:
            other_words |= words_per_sign[s2]
    exclusive = words_per_sign[sign] - other_words
    exclusive_per_sign[sign] = exclusive

print('=== Words exclusive to each zodiac SIGN ===\n')
for sign in SIGN_FOLIOS:
    print(f'{sign:<14}: {len(exclusive_per_sign[sign]):>3} exclusive words '
          f'(out of {len(words_per_sign[sign])})')

# Now: for each sign, find exclusive words that are length-compatible
# with the Hebrew or Aramaic zodiac name AND internally consistent
print('\n=== Exclusive + length-compatible candidates ===\n')

excl_candidates = {}
for sign in SIGN_FOLIOS:
    heb_name = HEBREW_ZODIAC[sign]
    ara_name = ARAMAIC_ZODIAC[sign]
    
    cands = []
    for w in sorted(exclusive_per_sign[sign]):
        freq = GLOBAL_FREQ.get(w, 0)
        for name, lang in [(heb_name, 'hebrew'), (ara_name, 'aramaic')]:
            m = check_mapping_consistency_many_to_one(list(w), name)
            if m is not None:
                cands.append({
                    'word': w, 'target': ''.join(name), 'lang': lang,
                    'mapping': m, 'global_freq': freq,
                })
    
    # Sort by frequency (rarest first)
    cands.sort(key=lambda x: x['global_freq'])
    excl_candidates[sign] = cands
    
    if cands:
        print(f'{sign:<14}: {len(cands)} exclusive candidates')
        for c in cands[:5]:
            w_str = ' '.join(c['word'])
            mp_str = ', '.join(f'{g}→{ch}' for g, ch in c['mapping'].items())
            print(f'    [{c["lang"]}] {w_str} → {c["target"]}  '
                  f'(freq={c["global_freq"]})  ({mp_str})')
    else:
        print(f'{sign:<14}: 0 exclusive candidates')
    print()

total_excl = sum(len(v) for v in excl_candidates.values())
print(f'Total exclusive candidates: {total_excl}')

# ── Greedy consistency on exclusive candidates ──
print(f'\n=== Greedy consistency (exclusive words only) ===\n')

sign_maps_excl = {}
for sign, cands in excl_candidates.items():
    if cands:
        sign_maps_excl[sign] = [(c['word'], c['target'], c['mapping'],
                                  c['lang']) for c in cands]

signs_excl = sorted(sign_maps_excl.keys())
print(f'Signs with exclusive candidates: {len(signs_excl)}')

best_global_excl = None
best_merged_excl = None
best_n_excl = 0

for start_sign in signs_excl:
    for start_cand in sign_maps_excl[start_sign]:
        current_mapping = dict(start_cand[2])
        selected = {start_sign: start_cand}
        
        remaining = sorted(
            [s for s in signs_excl if s != start_sign],
            key=lambda s: len(sign_maps_excl[s])
        )
        
        for next_sign in remaining:
            best_cand = None
            best_shared = -1
            for cand in sign_maps_excl[next_sign]:
                m = cand[2]
                shared = set(m.keys()) & set(current_mapping.keys())
                consistent = all(m[g] == current_mapping[g] for g in shared)
                if consistent and len(shared) > best_shared:
                    best_shared = len(shared)
                    best_cand = cand
            
            if best_cand is not None:
                selected[next_sign] = best_cand
                current_mapping.update(best_cand[2])
        
        if len(selected) > best_n_excl or (
            len(selected) == best_n_excl and
            len(current_mapping) > len(best_merged_excl or {})
        ):
            best_n_excl = len(selected)
            best_merged_excl = dict(current_mapping)
            best_global_excl = dict(selected)

if best_global_excl:
    print(f'\nBest: {best_n_excl} signs, {len(best_merged_excl)} mappings\n')
    for sign in signs_excl:
        if sign in best_global_excl:
            e = best_global_excl[sign]
            print(f'  {sign:<14}: {" ".join(e[0])} → {e[1]} [{e[3]}] '
                  f'(freq={GLOBAL_FREQ.get(e[0], 0)})')
        else:
            print(f'  {sign:<14}: (excluded)')
else:
    print('No consistent assignment found.')

# ── Monte Carlo ──
print(f'\n=== Monte Carlo (exclusive candidates, N=10,000) ===\n')

N_NULL_EXCL = 10_000
rng_excl = np.random.default_rng(777)

null_counts_excl = []
for sim in range(N_NULL_EXCL):
    shuffled = dict(zip(signs_list, rng_excl.permutation(signs_list)))
    
    sign_cands_null = {}
    for sign in signs_list:
        excl_words = exclusive_per_sign[sign]
        random_sign = shuffled[sign]
        names = zodiac_names_all[random_sign]
        cands = []
        for name_chars, lang in names:
            for w in excl_words:
                if len(w) != len(name_chars):
                    continue
                mapping = {}
                ok = True
                for g, c in zip(w, name_chars):
                    if g in mapping:
                        if mapping[g] != c:
                            ok = False
                            break
                    else:
                        mapping[g] = c
                if ok:
                    cands.append(mapping)
        if cands:
            sign_cands_null[sign] = cands
    
    n = greedy_max_consistent(sign_cands_null, signs_list)
    null_counts_excl.append(n)

null_counts_excl = np.array(null_counts_excl)
p_excl = np.mean(null_counts_excl >= best_n_excl)
mean_excl = np.mean(null_counts_excl)
max_excl = np.max(null_counts_excl)

print(f'Observed: {best_n_excl} consistent signs')
print(f'Null: mean={mean_excl:.2f}, max={max_excl}')
print(f'P(≥{best_n_excl} | null) = {p_excl:.4f}')
print(f'  → {"SIGNIFICANT (p<0.05)" if p_excl < 0.05 else "NOT significant"}')

fig, ax = plt.subplots(figsize=(8, 4))
bins = range(0, int(max_excl) + 3)
ax.hist(null_counts_excl, bins=bins, align='left', color='steelblue',
        edgecolor='white', density=True, label='Null (shuffled)')
ax.axvline(best_n_excl, color='red', linestyle='--', linewidth=2,
           label=f'Observed = {best_n_excl} (p={p_excl:.3f})')
ax.set_xlabel('Max consistent signs (exclusive words)')
ax.set_ylabel('Density')
ax.set_title('Zodiac Cribs: Exclusive Words Only')
ax.legend()
plt.tight_layout()
plt.savefig('images/zodiac_crib_exclusive_null.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/zodiac_crib_exclusive_null.png')

=== Words exclusive to each zodiac SIGN ===

Pisces        :  70 exclusive words (out of 114)
Aries         :  90 exclusive words (out of 151)
Taurus        : 106 exclusive words (out of 159)
Gemini        :  46 exclusive words (out of 94)
Cancer        :  91 exclusive words (out of 137)
Leo           :  71 exclusive words (out of 109)
Virgo         :  68 exclusive words (out of 92)
Libra         :  65 exclusive words (out of 93)
Scorpio       :  53 exclusive words (out of 84)
Sagittarius   :  59 exclusive words (out of 86)

=== Exclusive + length-compatible candidates ===

Pisces        : 13 exclusive candidates
    [hebrew] A3 B2 A1 C2 → דגים  (freq=1)  (A3→ד, B2→ג, A1→י, C2→ם)
    [aramaic] A3 B2 A1 C2 → נוני  (freq=1)  (A3→נ, B2→ו, A1→נ, C2→י)
    [aramaic] A3 B2 A3 F1 → נוני  (freq=1)  (A3→נ, B2→ו, F1→י)
    [aramaic] A3 Q2 A3 C1 → נוני  (freq=1)  (A3→נ, Q2→ו, C1→י)
    [hebrew] L1 K1 J1 A2 → דגים  (freq=1)  (L1→ד, K1→ג, J1→י, A2→ם)

Aries         : 13 exclusive candidates
    [he

/tmp/ipykernel_70529/2501683021.py:187: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ── Cell 11: Final summary ────────────────────────────────────────────

print('=' * 70)
print('ZODIAC CRIB ANALYSIS — FINAL SUMMARY')
print('=' * 70)

print(f'''
Zodiac pages: {len(ZODIAC_FOLIOS)} folios, 10 signs (Capricorn/Aquarius lost)
Total words: {total_labels} labels + {total_circ} circular text = {total_labels+total_circ}

=== Round 1: Labels only ===
  Candidates: {sum(len(v) for v in crib_candidates.values())}
  Best consistent set: {best_n_signs} signs, {len(best_merged)} mappings
  Monte Carlo p = {p_value:.4f} → NOT significant

=== Round 2: Labels + Circular text (freq ≤ {FREQ_THRESHOLD}) ===
  Candidates: {total_combined} total, {sum(len(v) for v in sign_maps_v2.values())} rare
  Best consistent set: {best_n_v2} signs, {len(best_merged_v2)} mappings
  Monte Carlo p = {p_value_v2:.4f} → NOT significant

=== Round 3: Exclusive words only (unique to one sign) ===
  Candidates: {total_excl}
  Best consistent set: {best_n_excl} signs, {len(best_merged_excl) if best_merged_excl else 0} mappings
  Monte Carlo p = {p_excl:.4f} → {"SIGNIFICANT" if p_excl < 0.05 else "NOT significant"}
''')

if best_global_excl:
    print('Best exclusive assignment:')
    for sign in signs_excl:
        if sign in best_global_excl:
            e = best_global_excl[sign]
            print(f'  {sign:<14}: {" ".join(e[0])} → {e[1]} [{e[3]}]')

print('''
CONCLUSION:
Three rounds of increasingly strict filtering all fail to produce
statistically significant cribs from the zodiac pages:
  - Round 1 (labels only): p=0.61
  - Round 2 (labels+Cc, rare): p=0.81  
  - Round 3 (exclusive words): p={p3:.2f}

The fundamental problem is combinatorial: even with exclusivity
constraints, the short target names (3-6 chars) produce too many
length-compatible candidates for cross-sign consistency to discriminate.

This does NOT prove the zodiac names are absent — only that a pure
substitution-based crib approach lacks statistical power here.
'''.format(p3=p_excl))

# Export final results
results_final = {
    'zodiac_folios': len(ZODIAC_FOLIOS),
    'signs_analyzed': list(SIGN_FOLIOS.keys()),
    'total_labels': total_labels,
    'total_circular_words': total_circ,
    'round1_labels_only': {
        'candidates': sum(len(v) for v in crib_candidates.values()),
        'consistent_signs': best_n_signs,
        'mapped_glyphs': len(best_merged) if best_merged else 0,
        'p_value': round(p_value, 4),
    },
    'round2_labels_and_cc': {
        'candidates_total': total_combined,
        'candidates_rare': sum(len(v) for v in sign_maps_v2.values()),
        'freq_threshold': FREQ_THRESHOLD,
        'consistent_signs': best_n_v2,
        'mapped_glyphs': len(best_merged_v2) if best_merged_v2 else 0,
        'p_value': round(p_value_v2, 4),
    },
    'round3_exclusive': {
        'candidates': total_excl,
        'consistent_signs': best_n_excl,
        'mapped_glyphs': len(best_merged_excl) if best_merged_excl else 0,
        'p_value': round(p_excl, 4),
    },
    'verdict': 'not_significant',
}

with open('zodiac_crib_resultados.json', 'w') as f:
    json.dump(results_final, f, indent=2, ensure_ascii=False)
print('Results saved to zodiac_crib_results.json')
print('=' * 70)

ZODIAC CRIB ANALYSIS — FINAL SUMMARY

Zodiac pages: 12 folios, 10 signs (Capricorn/Aquarius lost)
Total words: 341 labels + 910 circular text = 1251

=== Round 1: Labels only ===
  Candidates: 98
  Best consistent set: 4 signs, 14 mappings
  Monte Carlo p = 0.6103 → NOT significant

=== Round 2: Labels + Circular text (freq ≤ 3) ===
  Candidates: 412 total, 216 rare
  Best consistent set: 5 signs, 17 mappings
  Monte Carlo p = 0.8066 → NOT significant

=== Round 3: Exclusive words only (unique to one sign) ===
  Candidates: 193
  Best consistent set: 5 signs, 17 mappings
  Monte Carlo p = 0.7713 → NOT significant

Best exclusive assignment:
  Aries         : A1 Q1 N1 C2 → אמרא [aramaic]
  Leo           : A1 Q2 K1 C2 → אריא [aramaic]
  Pisces        : B1 A3 E1 B3 → דגים [hebrew]
  Sagittarius   : L1 J1 Ba A2 → קשתא [aramaic]
  Scorpio       : Aa B2 Q2 K2 A2 → עקרבא [aramaic]

CONCLUSION:
Three rounds of increasingly strict filtering all fail to produce
statistically significant cribs fr